<a href="https://colab.research.google.com/github/norasaleh1/Data-Science-Project/blob/main/Untitled16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#امبورت حق السيلينيوم
!pip install selenium
!pip install selenium webdriver_manager

import csv
import os
import pandas as pd
import time
import requests
from bs4 import BeautifulSoup
import pathlib
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=chrome_options)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 12.8 MB/s eta 0:00:00


In [9]:
import requests, pandas as pd, pathlib, re
from bs4 import BeautifulSoup
from datetime import datetime

BOE_URL = "https://laws.boe.gov.sa/BoeLaws/Laws/LawDetails/08381293-6388-48e2-8ad2-a9a700f2aa94/1"

def extract_boe_laborlaw(url=BOE_URL, save_raw=True, from_local_file=None):
    # 1) المصدر: ملف محلي أو طلب ويب
    if from_local_file:
        html_text = pathlib.Path(from_local_file).read_text(encoding="utf-8", errors="ignore")
        src_label = from_local_file
    else:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60, verify=False)
        r.raise_for_status()
        r.encoding = r.encoding or "utf-8"
        html_text = r.text
        src_label = url

    # (اختياري) حفظ HTML الخام
    raw_file = None
    if save_raw:
        raw_dir = pathlib.Path("data/raw/boe"); raw_dir.mkdir(parents=True, exist_ok=True)
        raw_file = raw_dir / f"laborlaw_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
        raw_file.write_text(html_text, encoding="utf-8")

    # 2) Parse
    soup = BeautifulSoup(html_text, "lxml")
    law_container = soup.select_one("#divLawText")
    if not law_container:
        raise RuntimeError("لم يتم العثور على العنصر #divLawText داخل الصفحة/الملف.")

    rows = []
    current_section = None   # الباب
    current_chapter = None   # الفصل

    for tag in law_container.select("*"):
        t = tag.get_text(strip=True)

        # تتبع عناوين الباب/الفصل العامة
        if tag.name == "h3" and t.startswith("الباب"):
            current_section = t
            continue
        if tag.name == "h3" and t.startswith("الفصل"):
            current_chapter = t
            continue

        # كل مادة داخل article_item
        if tag.name == "div" and "article_item" in (tag.get("class") or []):
            # جميع h3.center داخل نفس المادة
            h3_texts = [h.get_text(strip=True) for h in tag.find_all("h3", class_="center")]

            # لو الفصل مذكور داخل نفس المادة نحدّث current_chapter
            chapter_in_item = next((x for x in h3_texts if x.strip().startswith("الفصل")), None)
            if chapter_in_item:
                current_chapter = chapter_in_item

            # رقم المادة: أول نص يبدأ بـ "المادة"
            article_number = next((x for x in h3_texts if x.strip().startswith("المادة")), None)

            # لو ما انمسكت لأي سبب، جرّبي أي عنصر نصه يطابق ^المادة
            if not article_number:
                any_matn = tag.find(string=re.compile(r"^\s*المادة"))
                if any_matn:
                    article_number = any_matn.strip()

            # نص المادة
            text_tag = tag.find("div", class_="HTMLContainer")
            article_text = text_tag.get_text(" ", strip=True) if text_tag else None

            # حالة المادة
            css_classes = tag.get("class") or []
            status = "عادية"
            if "changed-article" in css_classes: status = "معدلة"
            if "canceled" in css_classes: status = "ملغاة"

            rows.append({
                "Section": current_section,
                "Chapter": current_chapter,
                "Article": article_number,
                "Text": article_text,
                "Status": status,
                "Source": src_label
            })

    df = pd.DataFrame(rows)

    # ✅ تحقق سريع: لا يسمح بأن يبدأ عمود Article بـ "الفصل"
    bad = df["Article"].fillna("").str.startswith("الفصل")
    if bad.any():
        # حاول إصلاح الصفوف السيئة عبر إعادة التقاط "المادة" من العمود النصي
        fixed = []
        for i, row in df[bad].iterrows():
            # استخرج "المادة ..." من نص المادة إن وُجد
            m = re.search(r"(المادة\s+\S+)", (row["Text"] or ""))
            fixed.append((i, m.group(1) if m else None))
        for i, val in fixed:
            if val:
                df.at[i, "Article"] = val

    # إن بقيت صفوف بدون مادة صحيحة، نبهّي المستخدم
    still_bad = df["Article"].isna() | df["Article"].str.startswith("الفصل", na=False)
    if still_bad.any():
        print("⚠️ صفوف ما زالت بدون 'المادة' بشكل صحيح. راجعي البنية لتلك المواد:", df[still_bad].head(3))

    out_dir = pathlib.Path("data/processed"); out_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_dir / "boe_laborlaw_articles.csv"
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"✅ Extracted {len(df)} articles")
    print(f"📄 CSV saved at: {out_csv}")
    if raw_file:
        print(f"🗂️ Raw HTML saved at: {raw_file}")

    return df


In [10]:
# Run extraction
df_boe = extract_boe_laborlaw(BOE_URL, save_raw=True)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'laws.boe.gov.sa'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Extracted 248 articles
📄 CSV saved at: data/processed/boe_laborlaw_articles.csv
🗂️ Raw HTML saved at: data/raw/boe/laborlaw_20251013_151723.html


In [8]:
from google.colab import files
files.download("/content/data/processed/boe_laborlaw_articles.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import files
files.download("/content/data/processed/boe_laborlaw_articles.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>